# 04 - SharePoint: indexed ingestion with document-level permissions

Index a SharePoint library into Azure AI Search with a SharePoint indexer, Text Split and ACL-preserving index projections, using direct REST calls and no Vision skills. The same content is queried without an index in **notebook 05 (remote)**.

| What runs | Where content lives | App registration |
|---|---|---|
| SharePoint indexer -> Text Split -> ACL-preserving index projections -> Search index | A searchable copy in Azure AI Search | Required for ingestion and native SharePoint group resolution |

This lab teaches Entra user/group ACLs **and native SharePoint site groups** (Owners, Members, Visitors). It is not a replacement for testing your source's permission model.

**Run order:** run section 1. The facilitator completes section 2 once (`scripts\setup-sharepoint-app.ps1` automates most of it); then run sections 3-6. Cleanup is opt-in. Authoring this notebook does not create an app or grant permissions.

## Prerequisites and safe test content

- Use the workshop tenant and a dedicated **non-sensitive test SharePoint site**, not a production library. Search and SharePoint should be in the same tenant for this lab. Sign in with `az login --tenant <tenant-id>`.
- Complete the [SharePoint indexer preview registration](https://aka.ms/azure-cognitive-search/indexer-preview). Review current preview limitations, including Conditional Access incompatibilities, unsupported guest-user ACLs and unsupported Entra groups nested inside SharePoint groups. Do not disable tenant security policies to make a lab run.
- Your user needs Search Service Contributor and Search Index Data Reader or Contributor. The workshop grant script supplies Contributor. The facilitator additionally needs permission to register applications, an appropriately privileged administrator for admin consent, and a SharePoint administrator/site-permission grant workflow. Azure subscription Owner alone is not tenant admin consent.
- In the test site's default Documents library, create two Word documents using the synthetic text in `data/search`: a general onboarding document and a restricted travel-policy document. Use direct users or direct membership in native SharePoint groups for the permission exercise. Avoid real employee or customer data. Both users must be able to access the site, but only one should have access to the restricted document.
- Add `SHAREPOINT_SITE_URL` and a unique `WORKSHOP_PREFIX` to your untracked `.env`. Set the registration identifiers after section 2. Never paste access tokens or client secrets into cells, output, or chat.

These are preview APIs. Read the linked documentation again before delivering the class.

In [ ]:
import json
import os
import re
import time
from pathlib import Path
from urllib.parse import urlparse
from uuid import UUID

import requests
from azure.identity import AzureCliCredential, get_bearer_token_provider
from dotenv import load_dotenv

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
if not (root / 'infra' / 'main.bicep').is_file():
    raise RuntimeError('Open from the workshop root or notebooks directory.')
load_dotenv(root / '.env')
load_dotenv(root / '.env.azure')
required = ('AZURE_TENANT_ID', 'AZURE_SEARCH_ENDPOINT', 'SHAREPOINT_SITE_URL')
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise RuntimeError(f'Missing settings: {missing}. See .env.example.')
tenant_id = str(UUID(os.environ['AZURE_TENANT_ID']))
prefix = os.getenv('WORKSHOP_PREFIX') or input('Unique attendee prefix (e.g. alice01): ').strip()
if not re.fullmatch(r'[a-z][a-z0-9]{2,19}', prefix):
    raise ValueError('Use 3-20 lowercase letters/digits, starting with a letter.')
site_url = os.environ['SHAREPOINT_SITE_URL'].rstrip('/')
site = urlparse(site_url)
if (site.scheme != 'https' or not site.hostname or not site.hostname.endswith('.sharepoint.com')
        or site.username or site.password or site.port not in (None, 443)
        or site.query or site.fragment or any(c in site_url for c in ';"\r\n')):
    raise ValueError('Use a public-cloud SharePoint HTTPS site URL, not a file or sharing-link URL.')
search_endpoint = os.environ['AZURE_SEARCH_ENDPOINT'].rstrip('/')
api_version = '2026-08-01-preview'
index_name = f'{prefix}-sp-chunks'
data_source_name = f'{prefix}-sp-source'
skillset_name = f'{prefix}-sp-skills'
indexer_name = f'{prefix}-sp-indexer'
credential = AzureCliCredential(tenant_id=tenant_id)
search_token = get_bearer_token_provider(credential, 'https://search.azure.com/.default')
graph_token = get_bearer_token_provider(credential, 'https://graph.microsoft.com/.default')
arm_token = get_bearer_token_provider(credential, 'https://management.azure.com/.default')
print('Search:', search_endpoint)
print('SharePoint site:', site_url)
print('API version:', api_version)


## 2. Facilitator lab: register and authorize the indexed SharePoint application

This is part of the workshop, not an unexplained prerequisite. `scripts\setup-sharepoint-app.ps1` performs 2a, 2b and optional consent in one run; the manual steps below explain what it does. Complete it before creating the SharePoint data source. One facilitator-owned ingestion app can be shared by attendee indexes on this Search service; attendees do not need to create or consent their own apps.

### 2a. Create the application and understand its IDs

In **Microsoft Entra ID -> App registrations -> New registration**, create `iq-workshop-sharepoint-indexer` as **single tenant**. The secretless ingestion flow needs no redirect URI, no public-client flow and no client secret. Record the **Application (client) ID** and the app registration's **Object ID** in `.env` as `SHAREPOINT_APP_CLIENT_ID` and `SHAREPOINT_APP_OBJECT_ID`.

Alternatively, the following **opt-in** cell creates a new application and service principal using Microsoft Graph. It declares permissions but does **not** grant admin consent, create a federated credential or grant site access. Your signed-in CLI identity must have the Graph authorization required to create apps/service principals; a 403 is not fixed by giving yourself a subscription role. Use the portal instead if your tenant does not permit this CLI workflow. Do not create a second app if the portal app already exists.

### Required application permissions

| API surface | Application permissions | Purpose |
|---|---|---|
| Microsoft Graph | `Files.Read.All`, `Sites.Selected` | Document ingestion and ACL extraction, following the current Search ACL permission matrix |
| SharePoint | `Sites.Selected`, `User.Read.All` | Read site permissions and resolve native SharePoint group membership at query time |

**Scope warning:** `Sites.Selected` requires an explicit grant on each workshop site; consent alone is insufficient. However, the separately required **Graph `Files.Read.All` is tenant-wide file-read permission**. This documented combination is **not strict site-only isolation** of the app. Review this with the tenant administrator and use a dedicated test tenant/site. We do not request tenant-wide `Sites.FullControl.All`. Do not silently remove a required permission or claim that the site URL in the data source limits the app's underlying authorization.

In the portal, add these permissions under their **correct API surfaces**, select **Application permissions**, then have an authorized administrator choose **Grant admin consent**. Requested permissions and consented permissions are different states.

In [ ]:
CREATE_APP_REGISTRATION = False
if CREATE_APP_REGISTRATION:
    if os.getenv('SHAREPOINT_APP_CLIENT_ID'):
        raise RuntimeError('An app is already configured. Reuse it; do not create a duplicate.')
    required_access = []
    for resource_app_id, permission_names in [
        ('00000003-0000-0000-c000-000000000000', ['Files.Read.All', 'Sites.Selected']),
        ('00000003-0000-0ff1-ce00-000000000000', ['Sites.Selected', 'User.Read.All']),
    ]:
        response = requests.get(
            'https://graph.microsoft.com/v1.0/servicePrincipals',
            headers={'Authorization': f'Bearer {graph_token()}'},
            params={'$filter': f"appId eq '{resource_app_id}'", '$select': 'id,appId,appRoles'},
            timeout=(10, 90),
        )
        response.raise_for_status()
        resources = response.json()['value']
        if len(resources) != 1:
            raise RuntimeError(f'Expected one API service principal for {resource_app_id}.')
        roles = {role['value']: role['id'] for role in resources[0]['appRoles']
                 if role.get('isEnabled') and 'Application' in role['allowedMemberTypes']}
        required_access.append({
            'resourceAppId': resource_app_id,
            'resourceAccess': [{'id': roles[name], 'type': 'Role'} for name in permission_names],
        })
    response = requests.post(
        'https://graph.microsoft.com/v1.0/applications',
        headers={'Authorization': f'Bearer {graph_token()}'},
        json={'displayName': 'iq-workshop-sharepoint-indexer', 'signInAudience': 'AzureADMyOrg',
              'requiredResourceAccess': required_access}, timeout=(10, 90),
    )
    response.raise_for_status()
    app = response.json()
    os.environ['SHAREPOINT_APP_CLIENT_ID'] = app['appId']
    os.environ['SHAREPOINT_APP_OBJECT_ID'] = app['id']
    print('Record in .env before continuing:')
    print('SHAREPOINT_APP_CLIENT_ID=' + app['appId'])
    print('SHAREPOINT_APP_OBJECT_ID=' + app['id'])
    response = requests.post(
        'https://graph.microsoft.com/v1.0/servicePrincipals',
        headers={'Authorization': f'Bearer {graph_token()}'},
        json={'appId': app['appId']}, timeout=(10, 90),
    )
    response.raise_for_status()
    print('Service principal created. Admin consent and site authorization are still required.')
else:
    print('No app created. Use the portal walkthrough or explicitly enable this facilitator-only cell.')


### 2b. Trust Search's managed identity (no client secret)

1. Open the Search resource -> **Identity** -> **System assigned**. It is enabled by Bicep. Note its **Object (principal) ID**.
2. Open **Entra ID -> Enterprise applications**, find this managed identity by its principal ID and copy its **Application ID**. Save that as `SHAREPOINT_SEARCH_MI_CLIENT_ID` in `.env`.
3. On the **ingestion app registration**, open **Certificates & secrets -> Federated credentials -> Add credential**. Choose the **Managed Identity** scenario, select this Search system-assigned identity and save a named credential. Do not create a client secret.
4. Verify the trust points at the exact Search identity, not the original/deleted Foundry account, your user or the Foundry project's identity.

| Identifier | Used for |
|---|---|
| Ingestion app client ID | `ApplicationId` and `sharePointConnectorAppRegistration.applicationId` |
| Search managed identity client/application ID | `FederatedCredentialApplicationId` and `sharePointConnectorAppRegistration.federatedCredentialId` |
| Search managed identity principal/object ID | Subject trusted by the app's federated credential |
| Ingestion app object ID | Graph `/applications/{id}` operations only |
| Federated credential object's own ID | **Not** the `federatedCredentialId` value in the Search index |

### 2c. Grant the app access to the selected test site

An administrator must make an explicit site permission grant **in addition to Entra admin consent**. For this ACL-inspection lab, use a site-scoped `FullControl` grant on the dedicated test site so the connector can inspect permission assignments. This allows changes within that site; it is not merely content-read access. Do not substitute a production site. Content-only indexing has a smaller permission requirement and can use `Read`; this lab also reads ACLs and resolves site groups.

For example, an administrator with the appropriate PnP/Graph authorization can run this in a **separate PowerShell terminal** (replace all placeholders). PnP needs an approved admin client application; this is distinct from the ingestion app:

```powershell
Connect-PnPOnline -Url 'https://<tenant>.sharepoint.com/sites/<workshop-site>' `
  -Interactive -ClientId '<approved-admin-client-id>'
Grant-PnPEntraIDAppSitePermission -AppId '<ingestion-app-client-id>' `
  -DisplayName 'iq-workshop-sharepoint-indexer' `
  -Site 'https://<tenant>.sharepoint.com/sites/<workshop-site>' -Permissions FullControl
Get-PnPEntraIDAppSitePermission -AppIdentity '<ingestion-app-client-id>' `
  -Site 'https://<tenant>.sharepoint.com/sites/<workshop-site>'
```

PnP is optional administrator tooling and is not a Python dependency. If unavailable, follow the linked Microsoft Graph site-permission grant documentation through your approved admin workflow. Verify the actual grant before proceeding; an app registration and a successful admin-consent screen alone do not prove site access.

### 2d. Optional read-only trust check

The following facilitator-only cell validates the managed-identity identifier and federated trust via ARM and Graph. It requires permission to read the app's federated credentials. It does not grant consent or site access; check those separately in the portal and the site-grant output.

In [ ]:
VERIFY_FEDERATION = False
if VERIFY_FEDERATION:
    app_object_id = str(UUID(os.environ['SHAREPOINT_APP_OBJECT_ID']))
    expected_mi_client_id = str(UUID(os.environ['SHAREPOINT_SEARCH_MI_CLIENT_ID']))
    search_id = os.environ['AZURE_SEARCH_RESOURCE_ID']
    response = requests.get(
        f'https://management.azure.com{search_id}?api-version=2025-05-01',
        headers={'Authorization': f'Bearer {arm_token()}'}, timeout=(10, 90),
    )
    response.raise_for_status()
    mi_principal_id = response.json()['identity']['principalId']
    response = requests.get(
        f'https://graph.microsoft.com/v1.0/servicePrincipals/{mi_principal_id}',
        headers={'Authorization': f'Bearer {graph_token()}'},
        params={'$select': 'id,appId'}, timeout=(10, 90),
    )
    response.raise_for_status()
    if str(UUID(response.json()['appId'])) != expected_mi_client_id:
        raise RuntimeError('SHAREPOINT_SEARCH_MI_CLIENT_ID is not this Search managed identity.')
    response = requests.get(
        f'https://graph.microsoft.com/v1.0/applications/{app_object_id}',
        headers={'Authorization': f'Bearer {graph_token()}'},
        params={'$select': 'id,appId'}, timeout=(10, 90),
    )
    response.raise_for_status()
    if str(UUID(response.json()['appId'])) != str(UUID(os.environ['SHAREPOINT_APP_CLIENT_ID'])):
        raise RuntimeError('App object ID and ingestion app client ID do not match.')
    response = requests.get(
        f'https://graph.microsoft.com/v1.0/applications/{app_object_id}/federatedIdentityCredentials',
        headers={'Authorization': f'Bearer {graph_token()}'}, timeout=(10, 90),
    )
    response.raise_for_status()
    trusts = response.json()['value']
    if not any(trust.get('subject') == mi_principal_id
               and trust.get('issuer') == f'https://login.microsoftonline.com/{tenant_id}/v2.0'
               and 'api://AzureADTokenExchange' in trust.get('audiences', []) for trust in trusts):
        raise RuntimeError('No matching managed-identity federated trust exists on this app.')
    print('Identity IDs and trust match. Still verify admin consent and the selected-site grant.')
else:
    print('Read-only trust check skipped; the facilitator must verify the trust in the portal.')


## 3. Indexed path: create the SharePoint data source

After completing the registration, consent, federation and site grant, set `INDEXED_SETUP_CONFIRMED=True`. The marker is an explicit facilitator acknowledgement, **not an automated permissions test**. A successful indexer run and two-user permission exercise are still required.

The connection string contains only non-secret IDs. Its two application IDs have different meanings. Native SharePoint group resolution requires federation; the simpler client-secret path is deliberately not used here.

`defaultSiteLibrary` scopes this lab to the selected site's default document library. It does not crawl all tenant sites, lists, or site pages. All files in that test library can be indexed; use a dedicated library/site containing only workshop content.

In [ ]:
INDEXED_SETUP_CONFIRMED = False
if not INDEXED_SETUP_CONFIRMED:
    raise RuntimeError('Complete section 2, then explicitly acknowledge it before indexed ingestion.')
sp_app_client_id = str(UUID(os.environ['SHAREPOINT_APP_CLIENT_ID']))
sp_mi_client_id = str(UUID(os.environ['SHAREPOINT_SEARCH_MI_CLIENT_ID']))
if sp_app_client_id == sp_mi_client_id:
    raise ValueError('The ingestion app and Search managed identity must have different client IDs.')
connection_string = (
    f'SharePointOnlineEndpoint={site_url};ApplicationId={sp_app_client_id};'
    f'FederatedCredentialApplicationId={sp_mi_client_id};TenantId={tenant_id}'
)
print('Indexed path configured. No client secret is used.')


In [ ]:
data_source = {
    'name': data_source_name, 'type': 'sharepoint',
    'credentials': {'connectionString': connection_string},
    'container': {'name': 'defaultSiteLibrary'},
    'indexerPermissionOptions': ['userIds', 'groupIds'],
}
response = requests.put(
    f'{search_endpoint}/datasources/{data_source_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json=data_source, timeout=(10, 90),
)
response.raise_for_status()
print('SharePoint data source ready:', data_source_name)


### Index schema and native SharePoint groups

Permission filtering is enabled on the index. `UserIds` and `GroupIds` carry the ACL metadata. `SharePointSiteUrl` and `sharePointConnectorAppRegistration` let Search resolve the querying user's native SharePoint group membership. ACL fields are not retrievable in normal results.

This lab uses keyword search to keep indexing and permission behavior visible. It needs neither embeddings nor a model deployment.

In [ ]:
index = {
    'name': index_name,
    'permissionFilterOption': 'enabled',
    'sharePointConnectorAppRegistration': {
        'applicationId': sp_app_client_id, 'tenantId': tenant_id,
        'federatedCredentialId': sp_mi_client_id,
    },
    'fields': [
        {'name': 'chunk_id', 'type': 'Edm.String', 'key': True,
         'searchable': True, 'analyzer': 'keyword'},
        {'name': 'parent_id', 'type': 'Edm.String', 'filterable': True},
        {'name': 'title', 'type': 'Edm.String', 'searchable': True},
        {'name': 'url', 'type': 'Edm.String'},
        {'name': 'chunk', 'type': 'Edm.String', 'searchable': True},
        {'name': 'UserIds', 'type': 'Collection(Edm.String)', 'permissionFilter': 'userIds',
         'filterable': True, 'retrievable': False},
        {'name': 'GroupIds', 'type': 'Collection(Edm.String)', 'permissionFilter': 'groupIds',
         'filterable': True, 'retrievable': False},
        {'name': 'SharePointSiteUrl', 'type': 'Edm.String', 'sharepointSiteUrl': True,
         'filterable': False, 'retrievable': False},
    ],
}
response = requests.put(
    f'{search_endpoint}/indexes/{index_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json=index, timeout=(10, 90),
)
response.raise_for_status()
print('ACL-aware index ready:', index_name)


## 4. Text Split with ACL-preserving projections

As in the Blob lab, logical pages are character chunks, not physical page citations. **Every chunk must inherit the source ACLs.** With `skipIndexingParentDocuments`, copying permissions only through indexer field mappings is insufficient.

Use `/document/metadata_spo_site_url` for the native-group site URL. Some older samples use a different metadata name; that is not the current documented source field. We intentionally avoid knowledge stores, enrichment caches, debug sessions and custom Web API skills here: the SharePoint ACL preview documents limitations with those features.

In [ ]:
skillset = {
    'name': skillset_name,
    'skills': [{
        '@odata.type': '#Microsoft.Skills.Text.SplitSkill',
        'name': 'split-text', 'context': '/document',
        'textSplitMode': 'pages', 'maximumPageLength': 2000, 'pageOverlapLength': 300,
        'inputs': [{'name': 'text', 'source': '/document/content'}],
        'outputs': [{'name': 'textItems', 'targetName': 'chunks'}],
    }],
    'indexProjections': {
        'selectors': [{
            'targetIndexName': index_name, 'parentKeyFieldName': 'parent_id',
            'sourceContext': '/document/chunks/*',
            'mappings': [
                {'name': 'chunk', 'source': '/document/chunks/*'},
                {'name': 'title', 'source': '/document/metadata_spo_item_name'},
                {'name': 'url', 'source': '/document/metadata_spo_item_path'},
                {'name': 'UserIds', 'source': '/document/metadata_user_ids'},
                {'name': 'GroupIds', 'source': '/document/metadata_group_ids'},
                {'name': 'SharePointSiteUrl', 'source': '/document/metadata_spo_site_url'},
            ],
        }],
        'parameters': {'projectionMode': 'skipIndexingParentDocuments'},
    },
}
response = requests.put(
    f'{search_endpoint}/skillsets/{skillset_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json=skillset, timeout=(10, 90),
)
response.raise_for_status()
print('ACL-preserving skillset ready:', skillset_name)


## 5. Create and monitor the SharePoint indexer

The first run starts on creation. Updating an existing idle indexer in this cell explicitly requests a new run; wait if it is already running. The SharePoint preview registration, application permissions, admin consent, selected-site grant, federated credential and Search permissions must all be correct. A successful data-source PUT alone does not prove that ingestion can read the site. No schedule is set; reruns are explicit.

In [ ]:
response = requests.get(
    f'{search_endpoint}/indexers/{indexer_name}/status?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
)
indexer_exists = response.status_code != 404
previous_run_start = None
if indexer_exists:
    response.raise_for_status()
    previous_run = response.json().get('lastResult') or {}
    if previous_run.get('status') == 'inProgress':
        raise RuntimeError('The indexer is already running. Wait before updating it.')
    previous_run_start = previous_run.get('startTime')
indexer = {
    'name': indexer_name, 'dataSourceName': data_source_name,
    'targetIndexName': index_name, 'skillsetName': skillset_name,
    'parameters': {
        'maxFailedItems': 0, 'maxFailedItemsPerBatch': 0,
        'configuration': {'dataToExtract': 'contentAndMetadata'},
    },
}
response = requests.put(
    f'{search_endpoint}/indexers/{indexer_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json=indexer, timeout=(10, 90),
)
response.raise_for_status()
print('SharePoint indexer ready:', indexer_name)
if indexer_exists:
    response = requests.post(
        f'{search_endpoint}/indexers/{indexer_name}/run?api-version={api_version}',
        headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
    )
    response.raise_for_status()
    print('New run requested for the existing indexer.')


In [ ]:
for attempt in range(90):
    response = requests.get(
        f'{search_endpoint}/indexers/{indexer_name}/status?api-version={api_version}',
        headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
    )
    response.raise_for_status()
    status = response.json()
    last_run = status.get('lastResult') or {}
    state = last_run.get('status')
    print('Run:', state, '| processed:', last_run.get('itemsProcessed'),
          '| failed:', last_run.get('itemsFailed'))
    is_requested_run = previous_run_start is None or last_run.get('startTime') != previous_run_start
    if state == 'success' and is_requested_run:
        print('Warnings:', json.dumps(last_run.get('warnings', []), indent=2))
        if last_run.get('itemsFailed', 0) or last_run.get('errors'):
            raise RuntimeError(json.dumps(last_run, indent=2))
        break
    if (is_requested_run and state in ('transientFailure', 'persistentFailure', 'error')) or status.get('status') == 'error':
        raise RuntimeError(json.dumps(last_run or status, indent=2))
    if attempt < 89:
        time.sleep(10)
else:
    raise TimeoutError('Indexer did not complete within 15 minutes. Inspect its status before retrying.')


## 6. Query indexed content as the signed-in user

`Authorization` authenticates access to Search. `x-ms-query-source-authorization` supplies the end-user identity for document permission filtering. In this local lab both represent your CLI-signed-in user, but their purposes are different. Do not substitute an application-only token for the end-user token.

There is **no elevated-read header** in this lab. Search Index Data Contributor can perform elevated reads; do not use that privilege to claim that an ordinary user can access a document. For production query clients, use appropriately restricted Search roles.

An empty result is valid when no authorized content matches. It is not proof that the indexer failed or that all permissions are correct.

In [ ]:
question = 'travel reimbursement receipts'
response = requests.post(
    f'{search_endpoint}/indexes/{index_name}/docs/search?api-version={api_version}',
    headers={
        'Authorization': f'Bearer {search_token()}',
        'x-ms-query-source-authorization': search_token(),
    },
    json={'search': question, 'count': True, 'select': 'title,url,chunk', 'top': 5},
    timeout=(10, 90),
)
response.raise_for_status()
indexed_result = response.json()
print(json.dumps(indexed_result, indent=2))


### Prove permission trimming and refresh behavior

Use two **non-admin member accounts**, not guest users, and confirm their actual access by opening the restricted document in SharePoint. Give account A access directly or through a direct native SharePoint-group membership; account B must not have access via any other group, sharing link or site-admin privilege. Run the same query and index name for both accounts, signing out/in to Azure CLI and restarting the kernel between identities. Both need the appropriate Search query role; only the facilitator needs to create objects.

Expected: account A can retrieve the restricted travel document; account B cannot. Record document URLs and observed results without saving source content into the committed notebook. Repeat with an Entra group and with a native SharePoint group to distinguish them. Do not infer that the entire SharePoint permission model is supported from one passing example.

Next change document permissions, then opt in to run the indexer again. Item-level unique ACL changes refresh on successful runs. **Inherited site/library/folder ACL changes need an explicit permissions resync** (or a supported document reset) and then an indexer run. The following cell demonstrates the resync; it touches only this named indexer. Rerun the monitor and query cells afterwards.

Changing a skillset instead requires reprocessing content, not just an ACL resync. Indexed content/permissions have synchronization latency; remote retrieval follows Microsoft 365's own indexing and permission-propagation behavior, not instantaneous document scanning.

In [ ]:
RUN_AGAIN = False
RESYNC_PERMISSIONS = False
if RUN_AGAIN:
    response = requests.get(
        f'{search_endpoint}/indexers/{indexer_name}/status?api-version={api_version}',
        headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
    )
    response.raise_for_status()
    last_run = response.json().get('lastResult') or {}
    if last_run.get('status') == 'inProgress':
        raise RuntimeError('The indexer is already running. Wait for it to finish.')
    previous_run_start = last_run.get('startTime')
    if RESYNC_PERMISSIONS:
        response = requests.post(
            f'{search_endpoint}/indexers/{indexer_name}/resync?api-version={api_version}',
            headers={'Authorization': f'Bearer {search_token()}'},
            json={'options': ['permissions']}, timeout=(10, 90),
        )
        response.raise_for_status()
    response = requests.post(
        f'{search_endpoint}/indexers/{indexer_name}/run?api-version={api_version}',
        headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
    )
    response.raise_for_status()
    print('Run requested. Rerun monitor-indexer, then query-index.')
else:
    print('No additional indexer run or ACL resync requested.')


## 7. Optional Search-object cleanup and separate tenant cleanup

Enable the following cell to remove only this notebook's named Search objects. Missing objects (404) are explicitly allowed; authorization failures and other errors are not ignored.

**This does not remove the ingestion app, its enterprise application, federated credential, consent, selected-site grant or source files.** The facilitator must first check whether any other attendee indexes still use the shared app. After all indexed labs are finished, revoke the app's site grant using the approved SharePoint/Graph admin workflow, remove the federated trust/consented permissions or delete the dedicated workshop app and service principal, and remove only the workshop documents/site that were created for this exercise. Do not delete a pre-existing app or site. App registrations are tenant objects and are not deleted by deleting the Azure resource group.

Clear all notebook outputs before sharing; outputs can contain private SharePoint document text and URLs even though tokens are not printed.

In [ ]:
CLEAN_UP = False
if CLEAN_UP:
    for collection, name in [
        ('indexers', indexer_name), ('skillsets', skillset_name),
        ('indexes', index_name), ('datasources', data_source_name),
    ]:
        response = requests.delete(
            f'{search_endpoint}/{collection}/{name}?api-version={api_version}',
            headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
        )
        if response.status_code != 404:
            response.raise_for_status()
        print('Deleted or already absent:', name)
else:
    print('Search-object cleanup disabled. Tenant app/site cleanup is always a separate admin step.')


## References and sample lineage

The local `bcgbrain/notebooks` examples informed this lab. The multimodal sample's `src/backend/data_injestion/skills.py` demonstrates Layout -> image verbalization -> text embeddings as a future non-Vision extension; this introductory lab intentionally does not import its app or provision those extra dependencies.

- [SharePoint indexer and app registration](https://learn.microsoft.com/azure/search/search-how-to-index-sharepoint-online)
- [ACL permissions matrix, native SharePoint groups, identifier lookup and refresh](https://learn.microsoft.com/azure/search/search-indexer-sharepoint-access-control-lists)
- [Grant a site permission with Microsoft Graph](https://learn.microsoft.com/graph/api/site-post-permissions)
- [Query-time ACL enforcement](https://learn.microsoft.com/azure/search/search-query-access-control-rbac-enforcement)

Skill/API versions and preview access can change. This notebook does not promise complete SharePoint permission-model parity. For retrieval without an index, continue with notebook 05.